# IMPORTS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import pennylane as qml
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# DATA PREPROCESSING FUNCTIONS

In [ ]:
def normalize_data(data):
    """Scales data to the range [-1, 1]."""
    scaler = MinMaxScaler(feature_range=(-1, 1))
    scaled_data = scaler.fit_transform(data)
    return scaled_data, scaler

def create_sequences(data, input_seq_length=7, output_seq_length=3):
    """Creates sequences of inputs and corresponding outputs."""
    sequences, labels = [], []
    for i in range(len(data) - input_seq_length - output_seq_length + 1):
        seq = data[i:i + input_seq_length]
        label = data[i + input_seq_length:i + input_seq_length + output_seq_length]
        sequences.append(seq)
        labels.append(label)
    return np.array(sequences), np.array(labels)

# QUANTUM LSTM CLASS DEFINITION

In [ ]:
class zzfeatuermapQLSTM(nn.Module):
    """A Quantum LSTM model where gates are replaced by variational quantum circuits."""
    def __init__(self, input_size, hidden_size, n_qubits=4, n_qlayers=1, backend="default.qubit"):
        super(zzfeatuermapQLSTM, self).__init__()
        self.n_inputs = input_size
        self.hidden_size = hidden_size
        self.concat_size = self.n_inputs + self.hidden_size
        self.n_qubits = n_qubits
        self.n_qlayers = n_qlayers
        self.backend = backend

        # =============== TODO: TASK 2a - DEFINE WIRES AND DEVICES ===============
        # INSTRUCTIONS: Define unique wire names and devices for the input and output gates.
        self.wires_forget = [f"wire_forget_{i}" for i in range(self.n_qubits)]
        self.wires_input = # YOUR CODE HERE
        self.wires_update = [f"wire_update_{i}" for i in range(self.n_qubits)]
        self.wires_output = # YOUR CODE HERE

        self.dev_forget = qml.device(self.backend, wires=self.wires_forget)
        self.dev_input = # YOUR CODE HERE
        self.dev_update = qml.device(self.backend, wires=self.wires_update)
        self.dev_output = # YOUR CODE HERE
        # ========================================================================

        # =============== TODO: TASK 2b - DEFINE QUANTUM CIRCUITS ===============
        # INSTRUCTIONS: Create the _circuit_input and _circuit_output functions.
        # They should match the structure of _circuit_forget.
        def _circuit_forget(inputs, weights):
            qml.templates.IQPEmbedding(inputs, wires=self.wires_forget)
            qml.templates.BasicEntanglerLayers(weights, wires=self.wires_forget)
            return [qml.expval(qml.PauliZ(w)) for w in self.wires_forget]

        def _circuit_input(inputs, weights):
            # YOUR CODE HERE
            pass

        def _circuit_update(inputs, weights):
            qml.templates.IQPEmbedding(inputs, wires=self.wires_update)
            qml.templates.BasicEntanglerLayers(weights, wires=self.wires_update)
            return [qml.expval(qml.PauliZ(w)) for w in self.wires_update]

        def _circuit_output(inputs, weights):
            # YOUR CODE HERE
            pass
        # ========================================================================

        # Bind circuits to devices and create TorchLayers
        weight_shapes = {"weights": (n_qlayers, n_qubits)}
        self.qlayer_forget = qml.qnn.TorchLayer(qml.QNode(_circuit_forget, self.dev_forget, interface="torch"), weight_shapes)
        self.qlayer_input = qml.qnn.TorchLayer(qml.QNode(_circuit_input, self.dev_input, interface="torch"), weight_shapes)
        self.qlayer_update = qml.qnn.TorchLayer(qml.QNode(_circuit_update, self.dev_update, interface="torch"), weight_shapes)
        self.qlayer_output = qml.qnn.TorchLayer(qml.QNode(_circuit_output, self.dev_output, interface="torch"), weight_shapes)

        # Classical layers for pre- and post-processing
        self.clayer_in = nn.Linear(self.concat_size, n_qubits)
        self.clayer_out = nn.Linear(self.n_qubits, self.hidden_size)

    def forward(self, x, init_states=None):
        batch_size, seq_length, _ = x.size()
        hidden_seq = []

        if init_states is None:
            h_t = torch.zeros(batch_size, self.hidden_size, device=x.device)
            c_t = torch.zeros(batch_size, self.hidden_size, device=x.device)
        else:
            h_t, c_t = init_states

        for t in range(seq_length):
            x_t = x[:, t, :]
            v_t = torch.cat((h_t, x_t), dim=1)
            y_t = self.clayer_in(v_t)

            # =============== TODO: TASK 2c - COMPLETE FORWARD PASS LOGIC ===============
            # INSTRUCTIONS: Complete the standard LSTM update equations.
            f_t = # YOUR CODE HERE: Compute forget gate
            i_t = # YOUR CODE HERE: Compute input gate
            g_t = torch.tanh(self.clayer_out(self.qlayer_update(y_t)))
            o_t = # YOUR CODE HERE: Compute output gate

            c_t = # YOUR CODE HERE: Update cell state
            h_t = # YOUR CODE HERE: Update hidden state
            # ===========================================================================

            hidden_seq.append(h_t.unsqueeze(1))

        hidden_seq = torch.cat(hidden_seq, dim=1)
        return hidden_seq, (h_t, c_t)

# MODEL WRAPPER CLASS

In [ ]:
class LSTMRegressor(nn.Module):
    """A wrapper to select between Quantum or Classical LSTM and add a final prediction layer."""
    def __init__(self, input_dim, hidden_dim, output_dim=3, n_qubits=0, n_qlayers=1, backend='default.qubit'):
        super(LSTMRegressor, self).__init__()
        self.output_dim = output_dim
        if n_qubits > 0:
            print(f"Using Quantum LSTM on backend {backend}")
            self.lstm = zzfeatuermapQLSTM(input_dim, hidden_dim, n_qubits=n_qubits, n_qlayers=n_qlayers, backend=backend)
        else:
            print("Using Classical LSTM")
            self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        prediction = self.fc(lstm_out[:, -1, :]) # Use only the last hidden state for prediction
        return prediction

# Main Workflow: Data Loading, Training, and Evaluation


In [ ]:
# --- 1. Data Loading ---
# =============== TODO: TASK 1 - LOAD DATA ===============
# INSTRUCTIONS: Load the 'daily_max_temp_SDL.csv' file into a pandas DataFrame.
daily_max_temp = # YOUR CODE HERE

data = daily_max_temp['temp_c'].values.reshape(-1, 1)
data_normalized, scaler = normalize_data(data)

# Print some info
print(daily_max_temp.head())
print(f"\nTotal data points: {len(data)}")

In [ ]:
# --- 2. Sequence Creation & Splitting ---
input_seq_len = 7
output_seq_len = 3
X, y = create_sequences(data_normalized, input_seq_len, output_seq_len)

train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).squeeze(-1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).squeeze(-1)

print(f"Training sequences: {len(X_train)}")
print(f"Testing sequences: {len(X_test)}")

# Model Configuration

In [ ]:
# =============== TODO: TASK 3 & 4 - CONFIGURE HYPERPARAMETERS ===============
# Model Parameters
Qinput_dim = # YOUR CODE HERE
Qhidden_dim = # YOUR CODE HERE
Qn_qubits = # YOUR CODE HERE

# Training Parameters
num_epochs = # YOUR CODE HERE
learning_rate = 0.01

In [ ]:
# --- 3. Model Initialization ---
model = LSTMRegressor(
    input_dim=Qinput_dim,
    hidden_dim=Qhidden_dim,
    output_dim=output_seq_len,
    n_qubits=Qn_qubits
)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
loss_function = nn.MSELoss()

print(model)

In [ ]:
# --- 4. Training Loop ---
print("\nStarting model training...")
loss_history = []
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = loss_function(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}")

# Evaluation and Visualization

In [ ]:
# --- 5. Evaluation & Prediction ---
print("\nTraining complete. Evaluating model...")
model.eval()
with torch.no_grad():
    predictions_normalized = model(X_test_tensor).numpy()

# Inverse transform to get original temperature values
y_test_original = scaler.inverse_transform(y_test_tensor.numpy())
y_pred_original = scaler.inverse_transform(predictions_normalized)

In [ ]:
# --- 6. Visualization & Metrics ---
for day in range(output_seq_len):
    plt.figure(figsize=(12, 6))
    plt.plot(y_test_original[:, day], label=f'True Day {day+1}', marker='o')
    plt.plot(y_pred_original[:, day], label=f'Predicted Day {day+1}', linestyle='--', marker='x')
    plt.title(f'Temperature Prediction for Day {day+1}')
    plt.xlabel('Test Sequence Index')
    plt.ylabel('Temperature (°C)')
    plt.legend()
    plt.grid(True)
    plt.show()

    r2 = r2_score(y_test_original[:, day], y_pred_original[:, day])
    mae = mean_absolute_error(y_test_original[:, day], y_pred_original[:, day])
    print(f"\nMetrics for Day {day+1} Prediction:")
    print(f"  R² Score: {r2:.4f}")
    print(f"  Mean Absolute Error (MAE): {mae:.4f}")